# Projekt 03 (final): Nachfrage erklaeren und vorhersagen

**Szenario (Fortsetzung von Modul 02):** Die Betriebsleitung von Capital Bikeshare
war von deiner EDA begeistert. Jetzt will sie mehr:

1. **Erklaeren:** Wie stark haengt die Tagesnachfrage von Wetter und Kalender ab —
   in Zahlen? (→ Regression)
2. **Vorhersagen:** Wie viele Raeder werden morgen gebraucht? (→ Zeitreihen-Prognose)

**Daten: echt** — dieselben Bike-Sharing-Daten (UCI) wie in Modul 02, diesmal die
**Tagesebene** (`day.csv`, 731 Tage 2011-2012). Vorbereitung (einmalig):

```
python daten/download_daten.py
```

**Bezug zum Skript:** Abschnitte 1.1-1.2 (Regression), 2.1 (Zeitreihen),
2.2 (Feature Engineering, Leakage!).

## 1. Daten laden

Entnormierung wie gehabt (`temp*41` = Grad C usw. — Doku lesen zahlt sich aus).

**Aufgabe:** Lade `daten/day.csv` (`parse_dates=["dteday"]`) und lege die entnormierten Spalten an: `temp_c = temp*41`, `hum_pct = hum*100`, `wind_kmh = windspeed*67`. Ein erster Zeitreihen-Plot von `cnt` verschafft Ueberblick.


In [ ]:
# Dein Code hier. (Musterloesung: loesung/loesung.ipynb)


## 2. Einfache Regression: Nachfrage ~ Temperatur

**Aufgaben:**
1. Fitte `cnt ~ temp_c` mit `LinearRegression` (Achtung: sklearn erwartet X als
   2D — `df[["temp_c"]]`, nicht `df["temp_c"]`).
2. Gib Steigung, Achsenabschnitt und $R^2$ aus und formuliere die Steigung als Satz
   („pro Grad mehr ...").
3. Zeichne Scatter + Regressionsgerade.

**Selbstcheck:** Dein $R^2$ sollte zwischen 0,35 und 0,45 liegen (≈ $r^2$ aus Modul 02, dort war $r = 0{,}627$).


In [ ]:
# Dein Code hier. (Musterloesung: loesung/loesung.ipynb)


## 3. Residuenanalyse — dem Modell auf die Finger schauen

**Aufgabe:** Plotte die Residuen ($y - \hat{y}$) gegen die Temperatur.
Siehst du eine Kruemmung? Was sagt sie inhaltlich? (Skript 1.1: Residuenplot!)

In [ ]:
# Dein Code hier. (Musterloesung: loesung/loesung.ipynb)


**Befund:** Umgekehrte U-Form — bei milden Temperaturen unterschaetzt die Gerade,
an sehr heissen Tagen ueberschaetzt sie. Inhaltlich klar: **ab ~30 Grad macht Radfahren
keinen Spass mehr.** Eine Gerade kann „erst rauf, dann runter" nicht abbilden.

**Aufgabe:** Ergaenze `temp_c2 = temp_c**2` als Feature und fitte
`cnt ~ temp_c + temp_c2`. Vergleiche $R^2$ und berechne den **Scheitelpunkt**
$-\beta_1 / (2\beta_2)$ — die rechnerische Wohlfuehltemperatur.

**Selbstcheck:** Der Scheitelpunkt der Parabel (rechnerische Wohlfuehltemperatur) sollte zwischen 25 und 31 °C liegen.


In [ ]:
# Dein Code hier. (Musterloesung: loesung/loesung.ipynb)


## 4. Multiple Regression: das Erklaerungsmodell

**Aufgabe:** Fitte `cnt ~ temp_c + temp_c2 + hum_pct + wind_kmh + workingday + yr`
und interpretiere JEDEN Koeffizienten in einem Satz — **ceteris paribus** (Skript 1.2!).
`yr` ist dabei besonders interessant (0 = 2011, 1 = 2012): Was misst er?

In [ ]:
# Dein Code hier. (Musterloesung: loesung/loesung.ipynb)


**Musterinterpretation (mit deiner vergleichen):**

- `hum_pct` ≈ −32: pro Prozentpunkt Luftfeuchte ~32 Ausleihen weniger (bei sonst gleichen Bedingungen)
- `wind_kmh` ≈ −77: Wind schreckt deutlich ab
- `workingday` ≈ +98: Arbeitstage bringen auf Tagesebene etwas mehr Volumen (die Pendler!)
- `yr` ≈ +1893: **2012 lagen die Tage im Schnitt ~1900 Ausleihen ueber 2011** — das ist
  der Wachstumstrend als Zahl. Ohne `yr` wuerde dieser Trend in die anderen
  Koeffizienten „einsickern" (Omitted Variable Bias!)
- `temp_c`/`temp_c2` einzeln zu interpretieren ist sinnlos (Multikollinearitaet
  zwischen $x$ und $x^2$ ist hier gewollt) — sie wirken nur *gemeinsam* als Parabel.

**Warnung vor Kausal-Sprech:** „+98 an Arbeitstagen" ist eine *Beschreibung* der Daten,
kein Experiment. Fuer Kausalaussagen fehlen uns Confounder-Kontrolle und Randomisierung.

## 5. Prognose: morgen frueh wissen, was heute Abend fehlt

Jetzt der Wechsel von **erklaeren** zu **vorhersagen** — mit den Regeln aus Skript 2.1/2.2:

- **Lag-Features:** `lag1` (gestern), `lag7` (vor einer Woche), `roll7`
  (Mittel der letzten 7 Tage, um einen Tag versetzt — warum das `.shift(1)` braucht,
  ist die Leakage-Frage unten!)
- **Zeitlicher Split:** Training bis 30.09.2012, Test ab 01.10.2012 (92 Tage)
- **Baselines zuerst:** naiv (= gestern) und saisonal-naiv (= vor 7 Tagen)
- **Metrik:** MAE (mittlerer absoluter Fehler — direkt in „Raedern" interpretierbar)

**Aufgabe:** Baue die drei Lag-Features, splitte zeitlich, berechne die MAE der
beiden Baselines und trainiere dann eine `LinearRegression` auf
Wetter + Kalender + Lags. Schlaegt sie die Baselines?

**Erwartung:** Das Regressionsmodell (MAE ≈ 860 Raeder) schlaegt beide Baselines: naiv/gestern (≈ 950) und saisonal-naiv/vor 7 Tagen. Gib alle drei MAE-Werte aus und visualisiere Prognose vs. Realitaet.


In [ ]:
# Dein Code hier. (Musterloesung: loesung/loesung.ipynb)


In [ ]:
# Dein Code hier. (Musterloesung: loesung/loesung.ipynb)


**Schau auf die schlechtesten Tage:** Ende Oktober 2012 bricht die Nachfrage auf
fast Null ein — das ist **Hurrikan Sandy** (29./30.10.2012, Washington D.C. stand
still). Kein Modell der Welt sagt so etwas aus Lag-Features vorher. Reale Prognosen
brauchen deshalb externe Informationen (Wetterwarnungen, Events) — und ehrliche
Fehlerangaben, die solche Tage einschliessen.

## 6. Die Leakage-Demo: Warum der zeitliche Split nicht verhandelbar ist

**Aufgabe:** Trainiere dasselbe Modell mit einem **zufaelligen** 75/25-Split
(`train_test_split`, `random_state=0`) und vergleiche die Test-MAE mit deinem
zeitlichen Split. Erklaere den Unterschied. (Skript 2.2: Leakage!)

**Erwartung:** Der Zufalls-Split liefert eine scheinbar *bessere* MAE (≈ 625) — das ist ein Leakage-Artefakt. Der ehrliche Wert ist der zeitliche Split (≈ 860).


In [ ]:
# Dein Code hier. (Musterloesung: loesung/loesung.ipynb)


**Erklaerung:** Beim Zufalls-Split stehen Trainingstage zeitlich ZWISCHEN den
Testtagen — das Modell kennt fuer einen Testtag oft den Vortag und den Folgetag aus
dem Training, und die Lag-Features tragen deren Information direkt hinein. Die ~625
sind keine Prognoseleistung, sondern ein Messartefakt. Der ehrliche Wert ist der
zeitliche Split (~860): **So gut waeren wir wirklich gewesen, haetten wir ab Oktober
2012 jeden Tag prognostiziert.**

## 7. Fazit an die Betriebsleitung

**Aufgabe:** Wieder 4-5 Bullet-Points in Alltagssprache: Was treibt die Nachfrage
(mit Zahlen!), wie gut koennen wir morgen vorhersagen, wo sind die Grenzen?

<details><summary>Musterfazit</summary>

- Temperatur ist der staerkste Hebel: Nachfrage steigt bis ~28 Grad und faellt darueber wieder — Hitzetage sind KEINE Spitzentage.
- Wind und hohe Luftfeuchte druecken die Nachfrage spuerbar (~77 bzw. ~32 Ausleihen je Einheit).
- Das Geschaeft ist 2012 um im Schnitt ~1.900 Ausleihen/Tag gegenueber 2011 gewachsen — Kapazitaetsplanung muss diesen Trend fortschreiben.
- Unsere Tagesprognose liegt im Schnitt ~860 Raeder daneben (bei 2.000-8.000 Ausleihen/Tag) und schlaegt einfache Faustregeln („wie gestern": ~950).
- Grenzen: Extremereignisse (Hurrikan Sandy) sind aus Verlaufsdaten nicht vorhersagbar — fuer solche Tage braucht es Wetterwarnungen im Prozess.
</details>

## Geschafft — was du jetzt kannst

- Regression als Erklaerungswerkzeug: Koeffizienten ceteris paribus lesen,
  $R^2$ einordnen, Residuen pruefen, Nichtlinearitaet per Quadratterm einfangen
- Regression als Prognosewerkzeug: Lag-Features, zeitlicher Split, Baselines, MAE
- Leakage nicht nur benennen, sondern **in Zahlen vorfuehren**
- Modellgrenzen ehrlich kommunizieren (Sandy!)

**Bonusaufgaben:**
1. Sin/Cos-Kodierung des Monats (Skript 2.2) statt `yr` + Saison-Dummies — hilft es?
2. Berechne die MAE getrennt fuer Okt/Nov/Dez — wird die Prognose zum Jahresende schlechter? Warum koennte das sein?
3. Standardisiere die Features und vergleiche die Koeffizienten — welche Variable hat jetzt den groessten Betrag, und warum ist das die fairere Vergleichsbasis?

---

**Modul 03 ist damit abgeschlossen.** Weiter geht es mit Machine Learning 1 —
dort wird aus „Regression fitten" ein systematischer Werkzeugkasten mit
Validierung, Regularisierung und vielen weiteren Modellfamilien.